<a href="https://colab.research.google.com/github/Munana122/hausa-sentiment/blob/main/hausa_sentiment_afriberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/hausa-sentiment/src", exist_ok=True)
%cd /content/drive/MyDrive/hausa-sentiment
!pip install -q datasets transformers accelerate scikit-learn pandas sentencepiece

Mounted at /content/drive
/content/drive/MyDrive/hausa-sentiment


In [ ]:
!pip install -q datasets transformers accelerate evaluate scikit-learn pandas matplotlib streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 114.3 MB/s eta 0:00:00


In [ ]:
%%writefile src/data.py
import pandas as pd
from datasets import Dataset, DatasetDict, ClassLabel, Features, Value

BASE = "https://raw.githubusercontent.com/hausanlp/NaijaSenti/main/data/annotated_tweets"
CLASSES = ["positive", "neutral", "negative"]


def load_splits(lang="hau"):
    files = {"train": "train.tsv", "validation": "dev.tsv", "test": "test.tsv"}
    feats = Features({"tweet": Value("string"),
                      "label": ClassLabel(names=CLASSES)})
    out = {}
    for split, fname in files.items():
        df = pd.read_csv(f"{BASE}/{lang}/{fname}", sep="\t")
        df = df.dropna(subset=["tweet", "label"])
        df["label"] = df["label"].map({c: i for i, c in enumerate(CLASSES)})
        out[split] = Dataset.from_pandas(
            df[["tweet", "label"]], features=feats, preserve_index=False)
    ds = DatasetDict(out)
    for split in ds:
        counts = pd.Series(ds[split]["label"]).map(dict(enumerate(CLASSES)))
        print(split, len(ds[split]), counts.value_counts().to_dict())
    return ds


def label_names(ds):
    return ds["train"].features["label"].names


if __name__ == "__main__":
    ds = load_splits()
    print(label_names(ds))
    print(ds["train"][0])

Overwriting src/data.py


In [ ]:
!mkdir -p src

In [ ]:
!cd src && python data.py

train 14172 {'neutral': 4912, 'positive': 4687, 'negative': 4573}
validation 2677 {'neutral': 896, 'negative': 894, 'positive': 887}
test 5303 {'neutral': 1789, 'negative': 1759, 'positive': 1755}
['positive', 'neutral', 'negative']
{'tweet': '@user Da kudin da Arewa babu wani abin azo agani da yayi wa alummah allah ya isa yacucemu wlh yarikitamana kasa yarikitamana kasuwanci harkar ilimi harkar lfy hanyoyi babu lantarki dasuransu komai yalalace ga cinhanci da rashawa a fili ko ina a Nigeria jamiyaryar su tabataman mlm 😭🗣', 'label': 2}


In [ ]:
%%writefile src/baseline.py
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
from data import load_splits, label_names

TEXT = "tweet"
ds = load_splits()
names = label_names(ds)
tr, te = ds["train"].to_pandas(), ds["test"].to_pandas()

rows = []
for max_f in [2000, 5000, 20000]:
    for ngram in [(1, 1), (1, 2)]:
        vec = TfidfVectorizer(max_features=max_f, ngram_range=ngram)
        clf = LogisticRegression(max_iter=1000)
        clf.fit(vec.fit_transform(tr[TEXT]), tr["label"])
        pred = clf.predict(vec.transform(te[TEXT]))
        rows.append(dict(max_features=max_f, ngram=ngram,
                         acc=accuracy_score(te["label"], pred),
                         macro_f1=f1_score(te["label"], pred, average="macro")))
print(pd.DataFrame(rows))

Overwriting src/baseline.py


In [ ]:
!cd src && python baseline.py

train 14172 {'neutral': 4912, 'positive': 4687, 'negative': 4573}
validation 2677 {'neutral': 896, 'negative': 894, 'positive': 887}
test 5303 {'neutral': 1789, 'negative': 1759, 'positive': 1755}
   max_features   ngram       acc  macro_f1
0          2000  (1, 1)  0.724684  0.726189
1          2000  (1, 2)  0.718273  0.720194
2          5000  (1, 1)  0.735621  0.736853
3          5000  (1, 2)  0.729587  0.730225
4         20000  (1, 1)  0.727324  0.727900
5         20000  (1, 2)  0.729021  0.729079


In [ ]:
%%writefile src/finetune.py
import sys, numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score
from data import load_splits, label_names

TEXT = "tweet"
model_name = sys.argv[1]          # e.g. castorini/afriberta_base
lr = float(sys.argv[2])           # e.g. 3e-5
epochs = int(sys.argv[3])         # e.g. 5
out = sys.argv[4]                 # e.g. runs/afriberta_3e-5

ds = load_splits()
names = label_names(ds)
tok = AutoTokenizer.from_pretrained(model_name)

def enc(batch):
    return tok(batch[TEXT], truncation=True, max_length=128)

ds = ds.map(enc, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(names),
    id2label=dict(enumerate(names)), label2id={n: i for i, n in enumerate(names)})

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir=out, learning_rate=lr, num_train_epochs=epochs,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    weight_decay=0.01, seed=42, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=ds["train"],
                  eval_dataset=ds["validation"], processing_class=tok,
                  compute_metrics=metrics)
trainer.train()
print("TEST:", trainer.evaluate(ds["test"]))
trainer.save_model(out); tok.save_pretrained(out)

Overwriting src/finetune.py


In [ ]:
!nvidia-smi

Tue Oct 13 10:35:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!cd src && python finetune.py castorini/afriberta_base 3e-5 5 ../runs/afriberta_3e-5

Streaming output truncated to the last 5000 lines.
 38% 1691/4430 [04:35<06:04,  7.51it/s]

 38% 1693/4430 [04:35<05:40,  8.04it/s]

 38% 1695/4430 [04:35<06:14,  7.30it/s]

 38% 1696/4430 [04:35<06:27,  7.06it/s]

 38% 1697/4430 [04:36<06:37,  6.88it/s]

 38% 1698/4430 [04:36<06:20,  7.19it/s]

 38% 1699/4430 [04:36<06:14,  7.29it/s]

 38% 1700/4430 [04:36<05:57,  7.64it/s]

 38% 1701/4430 [04:36<05:44,  7.91it/s]

 38% 1703/4430 [04:36<05:23,  8.43it/s]

 38% 1704/4430 [04:36<05:40,  8.00it/s]

 38% 1705/4430 [04:37<05:38,  8.06it/s]

 39% 1706/4430 [04:37<05:53,  7.71it/s]

 39% 1707/4430 [04:37<05:38,  8.04it/s]

 39% 1709/4430 [04:37<05:40,  7.98it/s]

 39% 1710/4430 [04:37<05:35,  8.11it/s]

 39% 1711/4430 [04:37<05:47,  7.82it/s]

 39% 1712/4430 [04:37<05:40,  7.97it/s]

 39% 1713/4430 [04:38<05:25,  8.36it/s]

 39% 1714/4430 [04:38<05:57,  7.59it/s]

 39% 1715/4430 [04:38<06:14,  7.24it/s]

 39% 1717/4430 [04:38<05:38,  8.02it/s]

 39% 1718/4430 [04:38<05:34,  8.11it/s]

 39% 1

In [ ]:
!nvidia-smi

Tue Oct 13 10:50:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             35W /   70W |       0MiB /  15360MiB |     57%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!cd src && python finetune.py

Traceback (most recent call last):
  File "/content/drive/MyDrive/hausa-sentiment/src/finetune.py", line 8, in <module>
    model_name = sys.argv[1]          # e.g. castorini/afriberta_base
                 ~~~~~~~~^^^
IndexError: list index out of range


In [ ]:
!cd src && python finetune.py castorini/afriberta_base 3e-5 5 ../runs/afriberta_3e-5
!cd src && python finetune.py castorini/afriberta_base 1e-5 5 ../runs/afriberta_1e-5
!cd src && python finetune.py Davlan/afro-xlmr-base 2e-5 5 ../runs/afroxlmr_2e-5
!cd src && python finetune.py xlm-roberta-base 2e-5 5 ../runs/xlmr_2e-5

train 14172 {'neutral': 4912, 'positive': 4687, 'negative': 4573}
validation 2677 {'neutral': 896, 'negative': 894, 'positive': 887}
test 5303 {'neutral': 1789, 'negative': 1759, 'positive': 1755}
Map: 100% 14172/14172 [00:01<00:00, 13550.05 examples/s]
Map: 100% 2677/2677 [00:00<00:00, 14248.44 examples/s]
Map: 100% 5303/5303 [00:00<00:00, 9522.12 examples/s]
Loading weights: 100% 133/133 [00:00<00:00, 5226.91it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_base
Key                        | Status     | 
---------------------------+------------+-
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.de

In [ ]:
%%writefile src/evaluate.py
import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report
from data import load_splits

MODEL = "../runs/afriberta_3e-5"
ds = load_splits()
names = ds["test"].features["label"].names
te = ds["test"].to_pandas()

clf = pipeline("text-classification", model=MODEL, device=0)
out = clf(te["tweet"].tolist(), batch_size=64, truncation=True, max_length=128)

te["true"] = te["label"].map(dict(enumerate(names)))
te["pred"] = [o["label"] for o in out]
te["conf"] = [o["score"] for o in out]
te["n_words"] = te["tweet"].str.split().str.len()
te["wrong"] = te["true"] != te["pred"]
err = te[te["wrong"]]

print(classification_report(te["true"], te["pred"], labels=names, digits=3))
print(pd.crosstab(te["true"], te["pred"]).reindex(index=names, columns=names))
print(f"\n{len(err)} errors out of {len(te)}")
print("\nError rate by tweet length (words):")
print(te.groupby(pd.cut(te["n_words"], [-1, 5, 10, 20, 500]), observed=True)["wrong"].mean())
print("\nConfident errors (conf > 0.95):", (err["conf"] > 0.95).sum())

err.sample(40, random_state=1).to_csv("../errors_sample.csv", index=False)
err[err["conf"] > 0.95].head(20).to_csv("../confident_errors.csv", index=False)

Writing src/evaluate.py


In [ ]:
!cd src && python evaluate.py

train 14172 {'neutral': 4912, 'positive': 4687, 'negative': 4573}
validation 2677 {'neutral': 896, 'negative': 894, 'positive': 887}
test 5303 {'neutral': 1789, 'negative': 1759, 'positive': 1755}
Traceback (most recent call last):
  File "/content/drive/MyDrive/hausa-sentiment/src/evaluate.py", line 11, in <module>
    clf = pipeline("text-classification", model=MODEL, device=0)
  File "/usr/local/lib/python3.13/dist-packages/transformers/pipelines/__init__.py", line 1035, in pipeline
    model = load_model(
        adapter_path if adapter_path is not None else model,
    ...<4 lines>...
        **model_kwargs,
    )
  File "/usr/local/lib/python3.13/dist-packages/transformers/pipelines/base.py", line 276, in load_model
    raise ValueError(
        f"Could not load model {model} with any of the following classes: {class_tuple}. See the original errors:\n\n{error}\n"
    )
ValueError: Could not load model ../runs/afriberta_3e-5 with any of the following classes: (<class 'transformers.

In [3]:
!ls -la runs/afriberta_3e-5
!ls runs/afriberta_3e-5/checkpoint-* | head -20

total 4895
drwx------ 7 root root    4096 Oct 12 13:24 .
drwx------ 3 root root    4096 Oct 12 13:24 ..
drwx------ 2 root root    4096 Oct 12 13:28 checkpoint-1772
drwx------ 2 root root    4096 Oct 12 13:30 checkpoint-2658
drwx------ 2 root root    4096 Oct 12 13:33 checkpoint-3544
drwx------ 2 root root    4096 Oct 12 13:35 checkpoint-4430
drwx------ 2 root root    4096 Oct 12 13:26 checkpoint-886
-rw------- 1 root root    1002 Oct 12 13:35 config.json
-rw------- 1 root root     454 Oct 12 13:35 tokenizer_config.json
-rw------- 1 root root 4975887 Oct 12 13:35 tokenizer.json
-rw------- 1 root root    5201 Oct 12 13:35 training_args.bin
runs/afriberta_3e-5/checkpoint-1772:
config.json
model.safetensors
optimizer.pt
rng_state.pth
scheduler.pt
tokenizer_config.json
tokenizer.json
trainer_state.json
training_args.bin

runs/afriberta_3e-5/checkpoint-2658:
config.json
model.safetensors
optimizer.pt
rng_state.pth
scheduler.pt
tokenizer_config.json
tokenizer.json
trainer_state.json


In [4]:
import glob, json, os, shutil

run = "runs/afriberta_3e-5"
ckpts = sorted(glob.glob(f"{run}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))

# validation macro-F1 per step, from the training log
state = json.load(open(f"{ckpts[-1]}/trainer_state.json"))
val = {e["step"]: e["eval_macro_f1"] for e in state["log_history"] if "eval_macro_f1" in e}

usable = {}
for c in ckpts:
    step = int(c.split("-")[-1])
    f = f"{c}/model.safetensors"
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f"{os.path.basename(c)}: model file {size:.0f} MB | val macro-F1: {val.get(step)}")
    if size > 100:
        usable[c] = val.get(step, 0)

best = max(usable, key=usable.get)
print("\nUsing:", os.path.basename(best), "| val macro-F1:", usable[best])

os.makedirs("models/afriberta_best", exist_ok=True)
for fn in ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"]:
    shutil.copy(f"{best}/{fn}", f"models/afriberta_best/{fn}")
print(os.listdir("models/afriberta_best"))

checkpoint-886: model file 446 MB | val macro-F1: 0.789023117367516
checkpoint-1772: model file 446 MB | val macro-F1: 0.795273503055871
checkpoint-2658: model file 446 MB | val macro-F1: 0.7942157051290973
checkpoint-3544: model file 446 MB | val macro-F1: 0.7947841209978938
checkpoint-4430: model file 0 MB | val macro-F1: 0.7970367414437588

Using: checkpoint-1772 | val macro-F1: 0.795273503055871
['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json']


In [5]:
%%writefile src/evaluate.py
import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report
from data import load_splits

MODEL = "../models/afriberta_best"
ds = load_splits()
names = ds["test"].features["label"].names
te = ds["test"].to_pandas()

clf = pipeline("text-classification", model=MODEL, device=0)
out = clf(te["tweet"].tolist(), batch_size=64, truncation=True, max_length=128)

te["true"] = te["label"].map(dict(enumerate(names)))
te["pred"] = [o["label"] for o in out]
te["conf"] = [o["score"] for o in out]
te["n_words"] = te["tweet"].str.split().str.len()
te["wrong"] = te["true"] != te["pred"]
err = te[te["wrong"]]

print(classification_report(te["true"], te["pred"], labels=names, digits=3))
print(pd.crosstab(te["true"], te["pred"]).reindex(index=names, columns=names))
print(f"\n{len(err)} errors out of {len(te)}")
print("\nError rate by tweet length (words):")
print(te.groupby(pd.cut(te["n_words"], [-1, 5, 10, 20, 500]), observed=True)["wrong"].mean())
print("\nConfident errors (conf > 0.95):", (err["conf"] > 0.95).sum())

te.to_csv("../test_predictions.csv", index=False)
err.sample(40, random_state=1).to_csv("../errors_sample.csv", index=False)
err[err["conf"] > 0.95].head(20).to_csv("../confident_errors.csv", index=False)
te[~te["wrong"]].groupby("true").sample(10, random_state=1).to_csv("../correct_sample.csv", index=False)

Overwriting src/evaluate.py


In [11]:
import pandas as pd
pd.set_option("display.max_colwidth", 250)
d = pd.read_csv("test_predictions.csv")
nn = d[(d.true == "neutral") & (d.pred == "negative")]
print(len(nn))
nn.sample(25, random_state=1)[["tweet", "conf"]]

571


,tweet,conf
2575,shi wanna lockdown din dabaki ake jawosa,0.973014
2164,indai bata mutuba toh wallahi bai yiba maybe ma tsarki yazo yi matah,0.551406
3269,wai ma to ai sa gaya mana wane irin girma laifin tsigewa zai kai domin mu ilmanta,0.590503
2289,gsky zai wahala kano bamu dauki corona nigerian premier league bah we are the second,0.573647
1977,oh ni diyar bello mi ake da kudaden in an ciyo bashin su,0.911288
3092,haka yake so sinadarin zuciya mai kashe duk kan radadi,0.505004
3397,se suzo su kwace shi ai,0.971498
2248,ga dukkan alamu wani ya gayama gwamnatin filato cewa in an je lahira ana dawowa,0.814897
3019,burkinafaso congo algeria zambia ghana cameroon niger uganda saura wani yace qaryane,0.803289
2566,sauran ba mutane bah aljanai ne kenan tunda mutane sha daya neh bil adama,0.831391


In [7]:
!cd src && python evaluate.py

train 14172 {'neutral': 4912, 'positive': 4687, 'negative': 4573}
validation 2677 {'neutral': 896, 'negative': 894, 'positive': 887}
test 5303 {'neutral': 1789, 'negative': 1759, 'positive': 1755}
Loading weights: 100% 137/137 [00:00<00:00, 2636.24it/s]
              precision    recall  f1-score   support

    positive      0.864     0.846     0.855      1755
     neutral      0.796     0.601     0.685      1789
    negative      0.681     0.866     0.763      1759

    accuracy                          0.770      5303
   macro avg      0.780     0.771     0.767      5303
weighted avg      0.781     0.770     0.767      5303

pred      positive  neutral  negative
true                                 
positive      1484      130       141
neutral        143     1075       571
negative        90      146      1523

1221 errors out of 5303

Error rate by tweet length (words):
n_words
(-1, 5]      0.216194
(5, 10]      0.249466
(10, 20]     0.254811
(20, 500]    0.173913
Name: wrong, dtyp

In [8]:
import glob, json, os, shutil
run = "runs/afriberta_3e-5"
ckpts = sorted(glob.glob(f"{run}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
state = None
for c in reversed(ckpts):
    f = f"{c}/trainer_state.json"
    if os.path.exists(f):
        state = json.load(open(f)); break
best = os.path.basename(state["best_model_checkpoint"])
print("best:", best, "| val macro-F1:", state["best_metric"])
os.makedirs("models/afriberta_best", exist_ok=True)
for fn in ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"]:
    shutil.copy(f"{run}/{best}/{fn}", f"models/afriberta_best/{fn}")
print(os.listdir("models/afriberta_best"))

best: checkpoint-4430 | val macro-F1: 0.7970367414437588


FileNotFoundError: [Errno 2] No such file or directory: 'runs/afriberta_3e-5/checkpoint-4430/model.safetensors'